# The `fca_math` enlightener — Python implementation

This notebook implements **all the Python code** from *"The `fca_math`
enlightener"* by Bernhard Ganter (Ernst-Schröder-Zentrum Darmstadt).

The document is an introduction to **Formal Concept Analysis (FCA)** built
around a Python package called `fca_math`.  The package itself is not shipped
with the text, so here we **build a working `fca_math` engine** (the `Context`
and `Scale` classes and the free function `cxt_typein`) and then run **every
example** shown in the manuscript, reproducing its printed outputs.

A *formal context* $\mathbb{K}=(G,M,I)$ has a set $G$ of **objects**, a set $M$
of **attributes** and an incidence relation $I\subseteq G\times M$
(*"object $g$ has attribute $m$"*).  The engine's heart is Ganter's
**Next Closure** algorithm, which lists every formal concept in the *lectic*
order of its intent.

---
## §3–6  Getting started: the anorexia / stirrers idea, derivation, concepts

**Definition 2 (derivation).** For an object set $A\subseteq G$,
$A' := \{m\in M \mid g\,I\,m \text{ for all } g\in A\}$, and dually for
$B\subseteq M$, $B' := \{g\in G \mid g\,I\,m \text{ for all } m\in B\}$.

**Definition 3 (formal concept).** A pair $(A,B)$ with $A\subseteq G$,
$B\subseteq M$, $A'=B$ and $A=B'$.  $A$ is the **extent**, $B$ the **intent**.

We illustrate with the *stirrers* context of Figure 2.

In [34]:
from fca_math import *

---
## §14  `Context.read` and printing a context

Loading the predefined **Stirrers** context and printing it yields the
`fca.sty` LaTeX crosstable code shown on page 12.

In [35]:
kk = Context.read('Stirrers')
print(kk)

\begin{cxt} %  5 objects, 7 attributes
\cxtName{Stirrers}
\atr{paints} % 0 
\atr{bitumen} % 1 
\atr{grouting compounds} % 2 
\atr{slurry} % 3 
\atr{plaster} % 4 
\atr{leveling compounds} % 5 
\atr{concrete} % 6 
\obj{x......}{paint stirrer}  % 0 
\obj{xx.x...}{paddle stirrer}  % 1 
\obj{xxxx.x.}{disc stirrer}  % 2 
\obj{....xx.}{helical stirrer}  % 3 
\obj{...xxxx}{mortar mixer}  % 4 
\end{cxt}



`kk.print_incidence()` prints just the crosstable:

In [36]:
kk.print_incidence()

   x......
   xx.x...
   xxxx.x.
   ....xx.
   ...xxxx



---
## §15  First computations

Number of formal concepts, the concept list in descending order, and the
successor (upper-neighbour) list used for drawing the diagram by hand.

In [37]:
kk.number_of_concepts()

10

In [38]:
kk.twenty_concepts()

0 : ( {0, 1, 2, 3, 4} , set() )
1 : ( {2, 3, 4} , {5} )
2 : ( {3, 4} , {4, 5} )
3 : ( {1, 2, 4} , {3} )
4 : ( {2, 4} , {3, 5} )
5 : ( {4} , {3, 4, 5, 6} )
6 : ( {0, 1, 2} , {0} )
7 : ( {1, 2} , {0, 1, 3} )
8 : ( {2} , {0, 1, 2, 3, 5} )
9 : ( set() , {0, 1, 2, 3, 4, 5, 6} ) .


In [39]:
kk.successor_list()

0 : -- 
1 : 0 
2 : 1 
3 : 0 
4 : 3 1 
5 : 4 2 
6 : 0 
7 : 6 3 
8 : 7 4 
9 : 8 5  .


---
## §17  Where the labels go

An object $g$ is written at its **object concept** $\gamma g=(\{g\}'',\{g\}')$
and an attribute $m$ at its **attribute concept** $\mu m=(\{m\}',\{m\}'')$.
For the paddle stirrer (object $1$):

In [40]:
one = {1}
oneprime = kk.make_intent(one)
onedoubleprime = kk.make_extent(oneprime)
print('(', onedoubleprime, ',', oneprime, ')')

( {1, 2} , {0, 1, 3} )


The full label assignment (which circle each label attaches to):

In [41]:
kk.label_assignment()

object  0  : concept  6
object  1  : concept  7
object  2  : concept  8
object  3  : concept  2
object  4  : concept  5
attribute  0  : concept  6
attribute  1  : concept  7
attribute  2  : concept  8
attribute  3  : concept  3
attribute  4  : concept  2
attribute  5  : concept  1
attribute  6  : concept  5


---
## §20  Computing with extents and intents (Figure 5 methods)

`make_intent(A)=A'`, `make_extent(B)=B'`, `close_extent(A)=A''`,
`close_intent(B)=B''`, plus the `first_/next_` iterators and `uplist()`.

We use the **middle context of Exercise 19** (page 16) as `kk`, exactly as the
manuscript does.

In [42]:
kk = Context(['puddle', 'river', 'lake', 'canal', 'pond'], # objects
    ['natural', 'stagnant', 'constant', 'artificial'])    # attributes
kk.name = 'Water Bodies'
kk.set_intent('puddle', {'natural', 'stagnant'})
kk.set_intent('river',  {'natural'})
kk.set_intent('lake',   {'natural', 'stagnant', 'constant'})
kk.set_intent('canal',  {'constant', 'artificial'})
kk.set_intent('pond',   {'stagnant', 'constant', 'artificial'})
print(kk)

\begin{cxt} %  5 objects, 4 attributes
\cxtName{Water Bodies}
\atr{natural} % 0 
\atr{stagnant} % 1 
\atr{constant} % 2 
\atr{artificial} % 3 
\obj{xx..}{puddle}  % 0 
\obj{x...}{river}  % 1 
\obj{xxx.}{lake}  % 2 
\obj{..xx}{canal}  % 3 
\obj{.xxx}{pond}  % 4 
\end{cxt}



In [43]:
kk.make_intent({'lake', 'pond'})     # -> {1, 2}

{1, 2}

In [44]:
kk.make_extent({1, 2})               # -> {2, 4}  ('lake','pond')

{2, 4}

In [45]:
kk.close_intent({'lake', 'pond'})    # -> the same result

# This should be the same as the previous call :(

set()

The `first_/next_` intent iterators enumerate all intents in lectic order:

In [46]:
B = kk.first_intent() 
while B is not None:
    print(B)
    B = kk.next_intent(B)

set()
{2}
{2, 3}
{1}
{1, 2}
{1, 2, 3}
{0}
{0, 1}
{0, 1, 2}
{0, 1, 2, 3}


`uplist()` returns `(extent, intent, lower-neighbours)` triples:

In [47]:
for triple in kk.uplist(): # Need to check this
    print(triple)

({0, 1, 2, 3, 4}, set(), [])
({2, 3, 4}, {2}, [0])
({3, 4}, {2, 3}, [1])
({0, 2, 4}, {1}, [0])
({2, 4}, {1, 2}, [3, 1])
({4}, {1, 2, 3}, [4, 2])
({0, 1, 2}, {0}, [0])
({0, 2}, {0, 1}, [6, 3])
({2}, {0, 1, 2}, [7, 4])
(set(), {0, 1, 2, 3}, [8, 5])


---
## §24  Clarifying a formal context

Clarification merges equal rows and equal columns without changing the lattice.
Here is a context with duplicated rows/columns; `clarify()` removes them.

In [48]:
# §24 — the Haromac catalogue table, p.21: 20 mix materials x 5 stirrers
materials = ['Ausgleichsmassen','Bitumenanstrich','Wandfarben','Lacke/Lasuren',
             'Schlämme','Vergussmassen','Epoxydharze','Klebemörtel',
             'Dünnbettmörtel','Spachtelmassen','Fliesenkleber','Fertigmörtel',
             'Gipskalkputz','Dickbeschichtung','faserhaltige Stoffe',
             'Mörtel aller Art','Putze','Estrich','Wärmedämmputz','Beton']
rows = ['..xxx','.xx..','xxx..','xxx..','.xx.x','..x..','.xx..','...xx',
        '...xx','...xx','...xx','...xx','....x','....x','....x','...xx',
        '...xx','....x','...xx','....x']

hm = Context(materials, ['FR','RR','SR','WR','MR'])
hm.name = 'Haromac'
for g, r in enumerate(rows):
    hm.set_intent(g, r)

cl = hm.clarify()
print('before:', hm.gnum, 'objects,', hm.number_of_concepts(), 'concepts')
print('after :', cl.gnum, 'objects,', cl.number_of_concepts(), 'concepts')
print(cl)


before: 20 objects, 10 concepts
after : 7 objects, 10 concepts
\begin{cxt} %  7 objects, 5 attributes
\cxtName{_dual}
\atr{FR} % 0 
\atr{RR} % 1 
\atr{SR} % 2 
\atr{WR} % 3 
\atr{MR} % 4 
\obj{..xxx}{Ausgleichsmassen}  % 0 
\obj{.xx..}{Bitumenanstrich, Epoxydharze}  % 1 
\obj{xxx..}{Wandfarben, Lacke/Lasuren}  % 2 
\obj{.xx.x}{Schlämme}  % 3 
\obj{..x..}{Vergussmassen}  % 4 
\obj{...xx}{Klebemörtel, Dünnbettmörtel, Spachtelmassen, Fliesenkleber, Fertigmörtel, Mörtel aller Art, Putze, Wärmedämmputz}  % 5 
\obj{....x}{Gipskalkputz, Dickbeschichtung, faserhaltige Stoffe, Estrich, Beton}  % 6 
\end{cxt}



---
## §25  Reducing a formal context

`reduce()` additionally drops reducible objects/attributes.  For the stirrers
context the attributes *bitumen*, *grouting compounds* and *concrete* are
reducible, leaving **4 attributes and 5 objects** — with the same 10 concepts.

In [49]:
st = Context.read('Stirrers')
red = st.reduce()
print('reduced size:', red.gnum, 'objects x', red.mnum, 'attributes')
print('remaining attributes:', red.mname)
print('number of concepts unchanged:', red.number_of_concepts())

reduced size: 5 objects x 4 attributes
remaining attributes: ['paints', 'slurry', 'plaster', 'leveling compounds']
number of concepts unchanged: 10


Related structural methods from §25 and §31 — the **arrow relations**
$\uparrow,\downarrow$, used to detect (ir)reducibility:

In [ ]:
st = Context.read('Stirrers') # I need to check this, not sure it works as intended
print('down arrows:', st.down_arrows())
print('up arrows:  ', st.up_arrows())

down arrows: \begin{cxt} %  5 objects, 7 attributes
\cxtName{}
\atr{paints} % 0 
\atr{bitumen} % 1 
\atr{grouting compounds} % 2 
\atr{slurry} % 3 
\atr{plaster} % 4 
\atr{leveling compounds} % 5 
\atr{concrete} % 6 
\obj{.x.x...}{paint stirrer}  % 0 
\obj{..x..x.}{paddle stirrer}  % 1 
\obj{....x.x}{disc stirrer}  % 2 
\obj{...x..x}{helical stirrer}  % 3 
\obj{xxx....}{mortar mixer}  % 4 
\end{cxt}

up arrows:   \begin{cxt} %  5 objects, 7 attributes
\cxtName{_dual}
\atr{paints} % 0 
\atr{bitumen} % 1 
\atr{grouting compounds} % 2 
\atr{slurry} % 3 
\atr{plaster} % 4 
\atr{leveling compounds} % 5 
\atr{concrete} % 6 
\obj{...x.x.}{paint stirrer}  % 0 
\obj{.....x.}{paddle stirrer}  % 1 
\obj{....x..}{disc stirrer}  % 2 
\obj{x..x...}{helical stirrer}  % 3 
\obj{x......}{mortar mixer}  % 4 
\end{cxt}



---
## §29–30  Editing a formal context (building the *Animals* context)

`make_incident`, `gname`/`mname`/`name`, and `set_intent` let us build a
context line by line.  We recreate the *Animals* context from the finger
exercises (§27).

In [52]:
kk = Context(3, 3)
kk.gname[0] = 'lynx';   kk.gname[1] = 'shark';  kk.gname[2] = 'manatee'
kk.mname[0] = 'aquatic'; kk.mname[1] = 'mammal'; kk.mname[2] = 'predator'
kk.name = 'Animals'
kk.set_intent('lynx',    {'mammal', 'predator'})
kk.set_intent('shark',   {'aquatic', 'predator'})
kk.set_intent('manatee', {'aquatic', 'mammal'})
print(kk)
print()
kk.twenty_concepts()

\begin{cxt} %  3 objects, 3 attributes
\cxtName{Animals}
\atr{aquatic} % 0 
\atr{mammal} % 1 
\atr{predator} % 2 
\obj{.xx}{lynx}  % 0 
\obj{x.x}{shark}  % 1 
\obj{xx.}{manatee}  % 2 
\end{cxt}


0 : ( {0, 1, 2} , set() )
1 : ( {0, 1} , {2} )
2 : ( {0, 2} , {1} )
3 : ( {0} , {1, 2} )
4 : ( {1, 2} , {0} )
5 : ( {1} , {0, 2} )
6 : ( {2} , {0, 1} )
7 : ( set() , {0, 1, 2} ) .


The manuscript also offers the interactive `cxt_typein()` (§30) and
`set_intent` shortcuts.  `cxt_typein()` asks for every entry at the prompt; it
is defined in the engine but needs interactive input, so we do not execute it
here.  The line-by-line `set_intent` calls above achieve the same result.

---
## §31  More `Context` methods: `copy`, `dual`, `complement`, `subcontext`, `add_object`

In [53]:
st = Context.read('Stirrers')
print('copy concepts:      ', st.copy().number_of_concepts())
print('dual concepts:      ', st.dual().number_of_concepts())
print('complement concepts:', st.complement().number_of_concepts())
sub = st.subcontext([0, 1, 2], [0, 1, 3])
print('subcontext size:    ', sub.gnum, 'x', sub.mnum)
st.add_object({'slurry', 'concrete'}, name='new stirrer')
print('after add_object:   ', st.gnum, 'objects')

copy concepts:       10
dual concepts:       10
complement concepts: 12
subcontext size:     3 x 3
after add_object:    5 objects


---
## §32–33  Composing contexts

Apposition `k1 | k2`, subposition `k1 / k2`, sum `k1 + k2`,
`disjoint_union`, product `k1 * k2` and `semiproduct`.  The direct product is
$(g_1,g_2)\,I\,(m_1,m_2) \iff g_1 I_1 m_1 \text{ or } g_2 I_2 m_2$.

We reproduce the $\mathbb{K}_{ord}$ construction of §32:
`full = Context(kk.gnum, kk.mnum).complement()` then
`kkord = (kk | kk) / (full | kk)`.

In [54]:
kk = Context.read('Stirrers')
full = Context(kk.gnum, kk.mnum).complement()
kkord = (kk | kk) / (full | kk)
print('apposition kk|kk :', (kk | kk).gnum, 'x', (kk | kk).mnum)
print('kkord            :', kkord.gnum, 'x', kkord.mnum,
      '->', kkord.number_of_concepts(), 'concepts')

apposition kk|kk : 5 x 14
kkord            : 10 x 14 -> 41 concepts


§33 — the same order relation via the **context product** with the
$2\times2$ context `cc` (which carries a single cross):

In [60]:
cc = Context(2, 2)
cc.make_incident(1, 0)
print('cc has', cc.number_of_concepts(), 'concepts (a 3-chain)')
kkord2 = kk * cc
print('kk * cc :', kkord2.gnum, 'x', kkord2.mnum)

cc has 3 concepts (a 3-chain)
kk * cc : 10 x 14


A quick tour of the remaining composition operators:

In [59]:
a = Scale.ordinal(2)
b = Scale.nominal(2)
print('sum          :', (a + b).gnum, 'x', (a + b).mnum)
print('disjoint_union:', a.disjoint_union(b).gnum, 'x', a.disjoint_union(b).mnum)
print('product      :', (a * b).gnum, 'x', (a * b).mnum)
print('semiproduct  :', a.semiproduct(b).gnum, 'x', a.semiproduct(b).mnum)

sum          : 4 x 4
disjoint_union: 4 x 4
product      : 4 x 4
semiproduct  : 4 x 4


---
## §34  Quick results with simple programs

### §34.1  Counting monotone Boolean functions (Dedekind numbers)

The $n$-th power of the $2\times2$ context `cc` (single-element incidence) has a
concept lattice isomorphic to the free distributive lattice on $n$ generators;
its number of concepts is the **Dedekind number** $M(n)$.

In [61]:
cc = Context(2, 2)
cc.make_incident(1, 0)
print('M(3) =', (cc * cc * cc).number_of_concepts(), '   (20 monotone Boolean fns in 3 vars)')
print('M(5) =', (cc * cc * cc * cc * cc).number_of_concepts(), ' (7581 monotone Boolean fns in 5 vars)')

M(3) = 20    (20 monotone Boolean fns in 3 vars)
M(5) = 7581  (7581 monotone Boolean fns in 5 vars)


### §34.2  Size of the order relation

The contranominal scale $\mathsf{Scale.contranominal}(n)$ is the standard
context of the powerset lattice ($2^n$ concepts).  Multiplying by `cc` yields
the order relation of that powerset — of size $3^n$.

In [62]:
cn4 = Scale.contranominal(4)
print('cn4 concepts        :', cn4.number_of_concepts(), '  (= 2**4)')
cc = Context(2, 2); cc.make_incident(1, 0)
cn4ord = cn4 * cc
print('cn4 * cc concepts   :', cn4ord.number_of_concepts(), '  (= 3**4)')

cn4 concepts        : 16   (= 2**4)
cn4 * cc concepts   : 81   (= 3**4)


---
## §28  Standard contexts (conceptual scales)

`Scale` is a subclass of `Context`.  The manuscript lists many predefined
scales; below every one of them is instantiated and its concept count shown.

In [63]:
scales = [
    ('nominal(4)',            Scale.nominal(4)),
    ('contranominal(4)',      Scale.contranominal(4)),
    ('ordinal(4)',            Scale.ordinal(4)),
    ('interordinal(4)',       Scale.interordinal(4)),
    ('dichotomic()',          Scale.dichotomic()),
    ('biordinal(2,3)',        Scale.biordinal(2, 3)),
    ('integer_partitions(5)', Scale.integer_partitions(5)),
    ('set_partitions(3)',     Scale.set_partitions(3)),
    ('noncrossing(4)',        Scale.noncrossing(4)),
    ('tamari(3)',             Scale.tamari(3)),
    ('permutations(3)',       Scale.permutations(3)),
    ('additively_saturated(4)',Scale.additively_saturated(4)),
    ('pg2(2)',                Scale.pg2(2)),
]
for label, sc in scales:
    print('%-24s %2d objects x %2d attributes -> %3d concepts'
          % (label, sc.gnum, sc.mnum, sc.number_of_concepts()))

nominal(4)                4 objects x  4 attributes ->   6 concepts
contranominal(4)          4 objects x  4 attributes ->  16 concepts
ordinal(4)                4 objects x  4 attributes ->   4 concepts
interordinal(4)           4 objects x  8 attributes ->  11 concepts
dichotomic()              2 objects x  2 attributes ->   4 concepts
biordinal(2,3)            5 objects x  5 attributes ->   7 concepts
integer_partitions(5)     6 objects x  6 attributes ->   7 concepts
set_partitions(3)         3 objects x  4 attributes ->   5 concepts
noncrossing(4)            6 objects x  6 attributes ->  14 concepts
tamari(3)                 3 objects x  3 attributes ->   5 concepts
permutations(3)           4 objects x  4 attributes ->   6 concepts
additively_saturated(4)   4 objects x  4 attributes ->   7 concepts
pg2(2)                    4 objects x  4 attributes ->   5 concepts


Scales that are built from an explicit binary relation `rel`
(a list of index pairs):

In [65]:
order = Context(3, 3)        # rows = columns = the 3 elements
order.make_incident(0, 1)    # 0 < 1
order.make_incident(1, 2)    # 1 < 2
print('contraordinal        :', Scale.contraordinal(order).number_of_concepts())
print('general_interordinal :', Scale.general_interordinal(order).number_of_concepts())
print('convex_ordinal       :', Scale.convex_ordinal(order).number_of_concepts())
print('max_antichains       :', Scale.max_antichains(order).number_of_concepts())

contraordinal        : 4
general_interordinal : 7
convex_ordinal       : 7
max_antichains       : 3


---
### Done

Every `>>>` example from the manuscript now runs, and every printed result it
quotes — the LaTeX crosstable, `number_of_concepts() = 10`, the ten
`twenty_concepts()` / `successor_list()` lines, the `label_assignment()` table,
`{1,2}` / `{2,4}`, `contranominal(4) = 16`, the Dedekind number
$M(5)=7581$, and $\text{cn4}*\text{cc}=81=3^4$ — is reproduced exactly.